In [1]:
pip install notebook

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("PySpark-jupyter-demo") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/28 18:38:33 WARN Utils: Your hostname, LAPTOP-74T6CTQ8, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/28 18:38:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/28 18:38:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
data = [("Kia", "Niro", 2025), ("Toyota", "Rav4", 2019), ("BYD", "Atto3", 2024)]
df = spark.createDataFrame(data, ["Car", "Model", "Year"])
df.show()

+------+-----+----+
|   Car|Model|Year|
+------+-----+----+
|   Kia| Niro|2025|
|Toyota| Rav4|2019|
|   BYD|Atto3|2024|
+------+-----+----+



In [7]:
df_lines = spark.read.format("com.databricks.spark.csv").options(header='true') \
 .option("delimiter", ",").option("quote", '"') \
 .option("escape", '"').option("multiLine", True) \
 .load("simpsons/simpsons_script_lines.csv")
df_lines.createOrReplaceTempView("lines_table")
## print lines file schema
df_lines.printSchema()
## print all fields for first 5 rows
spark.sql("SELECT * from lines_table").show(5)


root
 |-- id: string (nullable = true)
 |-- episode_id: string (nullable = true)
 |-- number: string (nullable = true)
 |-- raw_text: string (nullable = true)
 |-- timestamp_in_ms: string (nullable = true)
 |-- speaking_line: string (nullable = true)
 |-- character_id: string (nullable = true)
 |-- location_id: string (nullable = true)
 |-- raw_character_text: string (nullable = true)
 |-- raw_location_text: string (nullable = true)
 |-- spoken_words: string (nullable = true)
 |-- normalized_text: string (nullable = true)
 |-- word_count: string (nullable = true)

+----+----------+------+--------------------+---------------+-------------+------------+-----------+--------------------+--------------------+--------------------+--------------------+----------+
|  id|episode_id|number|            raw_text|timestamp_in_ms|speaking_line|character_id|location_id|  raw_character_text|   raw_location_text|        spoken_words|     normalized_text|word_count|
+----+----------+------+-------------

In [9]:
## print the shortest 10 script lines:
spark.sql("""
    SELECT int(word_count) AS wc, raw_text 
    FROM lines_table 
    WHERE TRY_CAST(word_count AS INT) > 0 
    ORDER BY wc 
""").show(10, False)

+---+---------------------------------+
|wc |raw_text                         |
+---+---------------------------------+
|1  |Homer Simpson: Oh.               |
|1  |Homer Simpson: And?              |
|1  |Bart Simpson: Lewis?             |
|1  |Homer Simpson: (SHOCKED) Me?     |
|1  |Wendell Borton: Yayyyyyyyyyyyyyy!|
|1  |Lisa Simpson: Baboon!            |
|1  |Lisa Simpson: Yeah.              |
|1  |Lisa Simpson: No!                |
|1  |Homer Simpson: Oh.               |
|1  |Homer Simpson: Nuts.             |
+---+---------------------------------+
only showing top 10 rows


In [11]:
## count the scenes that took more than 10 minutes
spark.sql("""
    SELECT count(*) 
    FROM lines_table 
    WHERE TRY_CAST(timestamp_in_ms AS INT) > (10 * 60 * 1000)
""").show()

+--------+
|count(1)|
+--------+
|   86678|
+--------+



In [13]:
## print the first 50 locations that has the word "Springfield" in them, ignoring letters case
spark.sql("""
    SELECT DISTINCT raw_location_text 
    FROM lines_table 
    WHERE LOWER(raw_location_text) LIKE '%springfield%' 
    LIMIT 50
""").show(50, False)

[Stage 9:>                                                          (0 + 1) / 1]

+---------------------------------------+
|raw_location_text                      |
+---------------------------------------+
|Springfield Police Station             |
|Springfield YMCA                       |
|SPRINGFIELD REPUBLICAN PARTY CASTLE    |
|Springfield                            |
|Springfield Pond                       |
|Springfield Mall                       |
|SPRINGFIELD ANIMAL REFUGE              |
|Springfield Community Theater          |
|SPRINGFIELD - THE PAST                 |
|PRIMORDIAL SPRINGFIELD                 |
|SPRINGFIELD HOSPITAL HALLWAY           |
|SPRINGFIELD ELEMENTARY SCHOOL BUS      |
|Springfield Elementary School Band Room|
|Springfield Opry House                 |
|Springfield Swap Meet                  |
|SPRINGFIELD PIER                       |
|Springfield Air and Space Museum       |
|Springfield Trade Center               |
|DREAMLIKE SUBURBAN SPRINGFIELD STREET  |
|SPRINGFIELD MONUMENT PARK              |
|Lake Springfield                 

In [14]:
## load the locations dataset
df_locs = spark.read.option("header", "true").csv("simpsons/simpsons_locations.csv")

## create a view
df_locs.createOrReplaceTempView("locations_table")

## print 20 quotes that are located in any place that has Jerusalem in its name.
spark.sql("""
    SELECT 
        loc.name AS location_name, 
        line.raw_text AS quote
    FROM lines_table AS line
    JOIN locations_table AS loc 
      ON line.location_id = loc.id
    WHERE LOWER(loc.name) LIKE '%jerusalem%'
    LIMIT 20
""").show(20, False)

+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|location_name         |quote                                                                                                                                                                                     |
+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|JERUSALEM ROYAL PALACE|(JERUSALEM ROYAL PALACE: ext. jerusalem royal palace - plaza - day)                                                                                                                       |
|JERUSALEM ROYAL PALACE|Bart Simpson: I'm bored. Send in my jester.                                                                                     

In [15]:
## print first 20 most used locations with the count of lines spoken in them.
spark.sql("""
    SELECT 
        raw_location_text, 
        COUNT(*) AS line_count
    FROM lines_table
    WHERE raw_location_text IS NOT NULL
    GROUP BY raw_location_text
    ORDER BY line_count DESC
    LIMIT 20
""").show(20, False)

+-------------------------------+----------+
|raw_location_text              |line_count|
+-------------------------------+----------+
|Simpson Home                   |35059     |
|Springfield Elementary School  |7092      |
|Moe's Tavern                   |4628      |
|Springfield Nuclear Power Plant|3594      |
|Kwik-E-Mart                    |1476      |
|First Church of Springfield    |1416      |
|Simpson Living Room            |1378      |
|Springfield Street             |1334      |
|Springfield                    |1314      |
|Simpson Car                    |1239      |
|Flanders Home                  |1166      |
|Street                         |1124      |
|Springfield Town Hall          |1103      |
|Springfield Retirement Castle  |1049      |
|Burns Manor                    |998       |
|Springfield Mall               |833       |
|Simpson Kitchen                |816       |
|Courtroom                      |813       |
|Bart's Treehouse               |767       |
|Bart's Be

In [18]:
## find the seasons in which the average imdb rating was the highest.
## Print the seasons number, the number of episodes in each one and the average rating
## in a descending order from highest average rating to lowest.

## load the episodes table
df_episodes = spark.read.option("header", "true").csv("simpsons/simpsons_episodes.csv")
df_episodes.createOrReplaceTempView("episodes_table")

## find the seasons in which the average imdb rating was the highest.
spark.sql("""
    SELECT 
        season, 
        COUNT(id) AS episode_count, 
        AVG(CAST(imdb_rating AS FLOAT)) AS avg_rating
    FROM episodes_table
    GROUP BY season
    ORDER BY avg_rating DESC
""").show(35, False)

+------+-------------+------------------+
|season|episode_count|avg_rating        |
+------+-------------+------------------+
|5     |22           |8.336363554000854 |
|7     |25           |8.32400001525879  |
|6     |25           |8.312000102996826 |
|4     |22           |8.268181844191117 |
|8     |25           |8.22              |
|3     |24           |8.154166638851166 |
|2     |22           |8.040909073569559 |
|9     |25           |7.844000053405762 |
|1     |13           |7.80769238105187  |
|10    |23           |7.569565233976944 |
|12    |21           |7.361904780069987 |
|11    |22           |7.290909138592807 |
|13    |22           |7.140909043225375 |
|14    |22           |7.077272761951793 |
|15    |22           |7.045454545454546 |
|16    |21           |7.0428571701049805|
|18    |22           |7.000000043348833 |
|19    |20           |6.934999942779541 |
|20    |21           |6.895238058907645 |
|17    |22           |6.863636341961947 |
|25    |22           |6.8318181254

In [21]:
##my question
##which 5 characters have the most dialogue lines in the series?

spark.sql("""
    SELECT 
        raw_character_text, 
        COUNT(*) AS total_lines
    FROM lines_table
    WHERE raw_character_text IS NOT NULL
    GROUP BY raw_character_text
    ORDER BY total_lines DESC
    LIMIT 5
""").show(5, False)

+-------------------+-----------+
|raw_character_text |total_lines|
+-------------------+-----------+
|Homer Simpson      |29842      |
|Marge Simpson      |14159      |
|Bart Simpson       |13777      |
|Lisa Simpson       |11502      |
|C. Montgomery Burns|3172       |
+-------------------+-----------+

